# Production unlearning annotation pipeline

This notebook processes **one or multiple digitally born PDF documents**, extracts paragraph-like text blocks with page coordinates, labels each unique paragraph with the strongest provider-specific model and prompt strategy from the cleaned benchmark, computes inter-model reliability, creates consensus labels, exports review tables, and writes **editable PDF highlight annotations and popup comments** back onto copies of the source documents.

## Provider defaults selected from the duplicate/leakage-adjusted benchmark

| Provider | Model | Prompt strategy | Cleaned accuracy | Cleaned Yes-F1 |
|---|---|---|---:|---:|
| OpenAI | `gpt-5.6-terra` | Codebook + leakage-checked labeled examples without metadata | 72.5% | 80.7% |
| Anthropic | `claude-sonnet-5` | Codebook + leakage-checked labeled examples without metadata | 70.0% | 80.6% |
| Google | `gemini-3.1-flash-lite` | Direct paragraph classification | 71.7% | 81.2% |

The two few-shot strategies use a gold-example workbook, but this notebook **automatically removes exact matches, near matches, and same-document examples before any API call**. That prevents the test leakage found in the earlier 49-row experiment.

## Main outputs

- `extracted_paragraphs.csv`
- one resumable JSONL file per provider
- `model_predictions_long.csv`
- `model_predictions_wide.csv`
- `consensus_annotations.csv`
- `unlearning_annotation_results.xlsx`
- one `*_unlearning_annotated.pdf` per input PDF
- a ZIP archive containing the run outputs

> PDF text extraction is layout dependent. Pages with little or no embedded text are flagged for OCR/manual review rather than silently processed with unreliable OCR.

## 1. Install dependencies

Run this once in a fresh Colab runtime or environment. Restarting the runtime is usually unnecessary.

In [1]:
%pip install -q -U openai anthropic google-genai pymupdf openpyxl xlsxwriter pydantic python-dotenv tqdm statsmodels rapidfuzz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 998.3/998.3 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 39.4 MB/s eta 0:00:00


## 2. Imports and shared utilities

In [2]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from itertools import combinations
from typing import Literal, Optional
from getpass import getpass

import os
import re
import json
import time
import math
import hashlib
import shutil
import warnings

import numpy as np
import pandas as pd
import fitz  # PyMuPDF

from pydantic import BaseModel, ConfigDict, Field, ValidationError
from tqdm.auto import tqdm
from sklearn.metrics import cohen_kappa_score
from statsmodels.stats.inter_rater import fleiss_kappa

try:
    from rapidfuzz.fuzz import ratio as rapid_ratio
except Exception:
    from difflib import SequenceMatcher
    def rapid_ratio(a, b):
        return 100.0 * SequenceMatcher(None, a, b).ratio()

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 100)


def env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}


def clean_text(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).replace("\u00ad", "")
    text = re.sub(r"-\s*\n\s*", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_for_match(value) -> str:
    text = clean_text(value).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def sha256_text(value: str) -> str:
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def safe_slug(value: str, max_len: int = 70) -> str:
    slug = re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_")
    return (slug[:max_len] or "document")


def normalize_yes_no(value):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"yes", "true", "1", "y"}:
        return True
    if text in {"no", "false", "0", "n"}:
        return False
    return np.nan

print("Imports loaded.")

Imports loaded.


## 3. Project paths and run configuration

For Colab, upload the approved codebook/example workbook into `/content/` and place one or more PDFs inside `/content/unlearning_pipeline/input_documents/`.

The same workbook may supply both the `Codebook` sheet and the labeled `GPT Test` examples. Change the paths when using a different approved workbook.

In [3]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
DEFAULT_BASE = Path("/content/unlearning_pipeline") if IN_COLAB else Path.cwd() / "unlearning_pipeline"
BASE_DIR = Path(os.getenv("UNLEARNING_BASE_DIR", str(DEFAULT_BASE))).expanduser().resolve()

INPUT_DIR = BASE_DIR / "input_documents"
OUTPUT_DIR = BASE_DIR / "outputs"
RAW_DIR = OUTPUT_DIR / "raw_provider_results"
ANNOTATED_DIR = OUTPUT_DIR / "annotated_pdfs"

for folder in [BASE_DIR, INPUT_DIR, OUTPUT_DIR, RAW_DIR, ANNOTATED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CODEBOOK_WORKBOOK_PATH = Path(os.getenv(
    "UNLEARNING_CODEBOOK_PATH",
    str(BASE_DIR / "Unlearning_Codebook_Combined_Human_Test_Set.xlsx"),
)).expanduser().resolve()

EXAMPLES_WORKBOOK_PATH = Path(os.getenv(
    "UNLEARNING_EXAMPLES_PATH",
    str(CODEBOOK_WORKBOOK_PATH),
)).expanduser().resolve()

CODEBOOK_SHEET = os.getenv("UNLEARNING_CODEBOOK_SHEET", "Codebook")
EXAMPLES_SHEET = os.getenv("UNLEARNING_EXAMPLES_SHEET", "GPT Test")
PDF_PATTERN = os.getenv("UNLEARNING_PDF_PATTERN", "*.pdf")

PROMPT_VERSION = "production_v1_best_provider_strategy"
RUN_API_CALLS = env_bool("UNLEARNING_RUN_API_CALLS", True)
MOCK_MODE = env_bool("UNLEARNING_MOCK_MODE", False)  # Only for local smoke tests.
RESUME_FROM_JSONL = env_bool("UNLEARNING_RESUME", True)

# Paragraph extraction settings.
MIN_PARAGRAPH_CHARS = int(os.getenv("UNLEARNING_MIN_PARAGRAPH_CHARS", "70"))
MIN_PARAGRAPH_TOKENS = int(os.getenv("UNLEARNING_MIN_PARAGRAPH_TOKENS", "12"))
LOW_TEXT_PAGE_CHARS = int(os.getenv("UNLEARNING_LOW_TEXT_PAGE_CHARS", "80"))
MARGINAL_ZONE_FRACTION = float(os.getenv("UNLEARNING_MARGIN_FRACTION", "0.12"))
REPEATED_MARGIN_PAGE_FRACTION = float(os.getenv("UNLEARNING_REPEATED_MARGIN_FRACTION", "0.50"))

# Few-shot/leakage settings.
N_FEWSHOT_EXAMPLES = int(os.getenv("UNLEARNING_N_FEWSHOT_EXAMPLES", "6"))
FEWSHOT_SEED = int(os.getenv("UNLEARNING_FEWSHOT_SEED", "42"))
LEAKAGE_SIMILARITY_THRESHOLD = float(os.getenv("UNLEARNING_LEAKAGE_THRESHOLD", "0.92"))
EXCLUDE_SAME_DOCUMENT_EXAMPLES = env_bool("UNLEARNING_EXCLUDE_SAME_DOCUMENT_EXAMPLES", True)
MIN_SAFE_FEWSHOT_EXAMPLES = int(os.getenv("UNLEARNING_MIN_SAFE_EXAMPLES", "4"))

# API execution settings.
MAX_OUTPUT_TOKENS = int(os.getenv("UNLEARNING_MAX_OUTPUT_TOKENS", "800"))
REQUEST_SLEEP_SECONDS = float(os.getenv("UNLEARNING_REQUEST_SLEEP", "0.15"))
MAX_RETRY_ATTEMPTS = int(os.getenv("UNLEARNING_MAX_RETRIES", "4"))
MAX_UNIQUE_PARAGRAPHS = None  # Set to a small integer for an API smoke test.

# Annotation behavior.
ANNOTATE_ANY_POSITIVE_VOTE = True
ANNOTATE_UNANIMOUS_NO = False
MAX_COMMENT_CHARS = 6000

MODEL_CONFIGS = [
    {
        "provider": "openai",
        "display_name": "OpenAI",
        "model": os.getenv("UNLEARNING_OPENAI_MODEL", "gpt-5.6-terra"),
        "strategy": "definitions_examples_no_metadata",
        "api_key_env": "OPENAI_API_KEY",
        "reasoning_effort": "low",
    },
    {
        "provider": "anthropic",
        "display_name": "Anthropic",
        "model": os.getenv("UNLEARNING_ANTHROPIC_MODEL", "claude-sonnet-5"),
        "strategy": "definitions_examples_no_metadata",
        "api_key_env": "ANTHROPIC_API_KEY",
        "effort": "low",
    },
    {
        "provider": "gemini",
        "display_name": "Google",
        "model": os.getenv("UNLEARNING_GEMINI_MODEL", "gemini-3.1-flash-lite"),
        "strategy": "direct_no_context",
        "api_key_env": "GEMINI_API_KEY",
        "thinking_level": "minimal",
    },
]

print("Base directory:", BASE_DIR)
print("Input PDF directory:", INPUT_DIR)
print("Codebook workbook:", CODEBOOK_WORKBOOK_PATH)
print("Examples workbook:", EXAMPLES_WORKBOOK_PATH)
print("RUN_API_CALLS:", RUN_API_CALLS, "| MOCK_MODE:", MOCK_MODE)
pd.DataFrame(MODEL_CONFIGS)[["provider", "model", "strategy", "api_key_env"]]

Base directory: /content/unlearning_pipeline
Input PDF directory: /content/unlearning_pipeline/input_documents
Codebook workbook: /content/unlearning_pipeline/Unlearning_Codebook_Combined_Human_Test_Set.xlsx
Examples workbook: /content/unlearning_pipeline/Unlearning_Codebook_Combined_Human_Test_Set.xlsx
RUN_API_CALLS: True | MOCK_MODE: False


,provider,model,strategy,api_key_env
0,openai,gpt-5.6-terra,definitions_examples_no_metadata,OPENAI_API_KEY
1,anthropic,claude-sonnet-5,definitions_examples_no_metadata,ANTHROPIC_API_KEY
2,gemini,gemini-3.1-flash-lite,direct_no_context,GEMINI_API_KEY


### Optional Colab upload helper

Run the next cell only when the files are not already present. It uploads PDFs into the input folder and Excel files into the project folder.

In [4]:
def upload_inputs_in_colab():
    if not IN_COLAB:
        raise RuntimeError("This helper is only available in Google Colab.")
    from google.colab import files

    uploaded = files.upload()
    copied = []
    for filename, data in uploaded.items():
        suffix = Path(filename).suffix.lower()
        destination = INPUT_DIR / filename if suffix == ".pdf" else BASE_DIR / filename
        destination.write_bytes(data)
        copied.append(str(destination))
    print("Uploaded:")
    for path in copied:
        print(" -", path)
    return copied

# Example usage:
# upload_inputs_in_colab()

## 4. API keys from environment variables, `.env`, Colab Secrets, or secure input

The SDKs read their normal environment variables. This cell never prints key values.

In [5]:
load_dotenv(BASE_DIR / ".env")
load_dotenv(Path(".env"))


def ensure_api_key(env_name: str, prompt_if_missing: bool = True) -> bool:
    if os.getenv(env_name):
        return True

    # Google Colab Secrets, when available.
    try:
        from google.colab import userdata
        secret = userdata.get(env_name)
        if secret:
            os.environ[env_name] = secret
            return True
    except Exception:
        pass

    if prompt_if_missing:
        secret = getpass(f"Enter {env_name} (input hidden; press Enter to skip): ").strip()
        if secret:
            os.environ[env_name] = secret
            return True

    return False


if RUN_API_CALLS and not MOCK_MODE:
    missing = []
    for cfg in MODEL_CONFIGS:
        if not ensure_api_key(cfg["api_key_env"], prompt_if_missing=True):
            missing.append(cfg["api_key_env"])
    if missing:
        raise RuntimeError(
            "Missing API keys: " + ", ".join(missing) +
            ". Set them in the environment, .env, Colab Secrets, or rerun this cell and enter them securely."
        )

print({cfg["api_key_env"]: bool(os.getenv(cfg["api_key_env"])) for cfg in MODEL_CONFIGS})

{'OPENAI_API_KEY': True, 'ANTHROPIC_API_KEY': True, 'GEMINI_API_KEY': True}


## 5. Load the approved codebook and labeled examples

The codebook loader finds the real `Code / Definition / Detection Logic / Examples / Positive Clarification / Negative Clarification` header row even when the workbook has an objective statement above it.

In [6]:
EXPECTED_CODEBOOK_COLUMNS = [
    "Code",
    "Definition",
    "Detection Logic",
    "Examples",
    "Positive Clarification",
    "Negative Clarification",
]


def load_codebook(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Codebook workbook not found: {path}")

    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
    header_index = None
    for idx, row in raw.iterrows():
        values = [clean_text(x).lower() for x in row.tolist()[:8]]
        if "code" in values and "definition" in values:
            header_index = idx
            break

    if header_index is None:
        raise ValueError(
            f"Could not find a Code/Definition header row in {path.name} / {sheet_name}."
        )

    header = [clean_text(x) for x in raw.iloc[header_index].tolist()]
    df = raw.iloc[header_index + 1:].copy()
    df.columns = header
    df = df.loc[:, [c for c in EXPECTED_CODEBOOK_COLUMNS if c in df.columns]]

    for col in EXPECTED_CODEBOOK_COLUMNS:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].map(clean_text)

    df = df[df["Code"].ne("")].reset_index(drop=True)
    if df.empty:
        raise ValueError("The parsed codebook contains no coding rows.")
    return df[EXPECTED_CODEBOOK_COLUMNS]


def load_examples(path: Path, sheet_name: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Examples workbook not found: {path}")

    df = pd.read_excel(path, sheet_name=sheet_name)
    if "Text Content" not in df.columns:
        raise ValueError(f"{sheet_name} must contain a 'Text Content' column.")

    label_col = "Unlearning" if "Unlearning" in df.columns else "Original Unlearning"
    if label_col not in df.columns:
        raise ValueError(f"{sheet_name} must contain an Unlearning label column.")

    rename = {label_col: "example_unlearning"}
    if "Original Target" in df.columns and "Target" not in df.columns:
        rename["Original Target"] = "Target"
    if "Original Government Agency" in df.columns and "Government Agency" not in df.columns:
        rename["Original Government Agency"] = "Government Agency"
    if "Row ID" in df.columns and "Number" not in df.columns:
        rename["Row ID"] = "Number"
    if "Document Title" in df.columns and "Document" not in df.columns:
        rename["Document Title"] = "Document"

    df = df.rename(columns=rename).copy()
    df["Text Content"] = df["Text Content"].map(clean_text)
    df["example_unlearning"] = df["example_unlearning"].map(normalize_yes_no)
    df = df[df["Text Content"].ne("") & df["example_unlearning"].notna()].reset_index(drop=True)

    for col in ["Number", "Codes", "Target", "Government Agency", "Document", "Rationale"]:
        if col not in df.columns:
            df[col] = ""
        df[col] = df[col].map(clean_text)

    return df


def serialize_codebook(df: pd.DataFrame) -> str:
    blocks = []
    for _, row in df.iterrows():
        blocks.append("\n".join([
            f"Code: {row['Code']}",
            f"Definition: {row['Definition']}",
            f"Detection logic: {row['Detection Logic']}",
            f"Examples: {row['Examples']}",
            f"Positive clarification: {row['Positive Clarification']}",
            f"Negative clarification: {row['Negative Clarification']}",
        ]))
    return "\n\n".join(blocks)


codebook_df = load_codebook(CODEBOOK_WORKBOOK_PATH, CODEBOOK_SHEET)
examples_pool_df = load_examples(EXAMPLES_WORKBOOK_PATH, EXAMPLES_SHEET)
CODEBOOK_TEXT = serialize_codebook(codebook_df)

print("Codebook rows:", len(codebook_df))
print("Available labeled example rows:", len(examples_pool_df))
print("Example label counts:", examples_pool_df["example_unlearning"].value_counts().to_dict())
display(codebook_df.head())

Codebook rows: 14
Available labeled example rows: 49
Example label counts: {True: 39, False: 10}


,Code,Definition,Detection Logic,Examples,Positive Clarification,Negative Clarification
0,Step 1 -- Binary classification based on unlearning definition,,,,,
1,Unlearning (Yes/No),"Unlearning is the deliberate process by which a government agency discards misaligned assumptions, obsolete knowledge, or outdated routines in response to demonstrated failure ...","Does the text explicitly identify a prior policy, assumption, practice, or technical system as inadequate or harmful — AND call for its removal, replacement, or fundamental ret...","""[T]he Federal government should not rely on the traditional layered approach and instead should proactively provide its capabilities and assistance directly to those in need.""...","Code when the text: • Names a specific prior assumption, policy, practice, or system as the source of failure. • Explicitly calls for abandoning, replacing, dismantling, or fun...",DO NOT code if: • The text only documents what went wrong without calling for change ('the levees failed'). • The change is purely additive — new resources or capacity added wi...
2,STEP 2 — TARGET OF UNLEARNING (assign one per passage),,,,,
3,Target (Leadership),"The text questions or redefines the role, authority, or responsibilities of a specific leadership position (e.g., DHS Secretary, FEMA Director, POTUS) in disaster management — ...",Does the text call for redefining or redistributing leadership authority — not just replacing the person in the role?,"""[A] single individual directly responsible and accountable to the President must be designated to act as the central focal point to lead and coordinate the overall federal res...","Code when the text: • Calls into question whether an existing leadership structure is capable of managing catastrophic events. • Proposes redistributing, consolidating, or rede...","DO NOT code if: • The text recommends appointing a new person to an existing, unchanged role. • Minor reporting-line adjustments are proposed without questioning the underlying..."
4,"Target (laws, plans and policies)","The text calls for terminating, replacing, or fundamentally revising a law, regulation, plan, or formal policy that is now seen as harmful, unjust, or structurally inadequate.","Does the text identify a specific law, plan, or policy as harmful or obsolete — and call for its elimination or replacement?","""Removing statutory restrictions on DOD authority to activate Reserve units for catastrophic disaster relief."" — GAO (Existing statutory constraint identified as harmful; remov...","Code when the text: • Calls for eliminating or replacing a specific statute, executive order, plan, or doctrine. • Frames an existing legal or policy arrangement as a root caus...","DO NOT code if: • The text proposes reform or better implementation of an existing policy without questioning its underlying logic. • A new plan is added alongside an old one, ..."


## 6. Extract paragraph-like blocks and their PDF coordinates

The extractor:

- keeps page and line rectangles for later editable highlighting;
- removes repeated short headers and footers;
- flags low-text pages instead of silently OCRing them;
- hashes normalized text so duplicate passages are labeled only once but remain annotated at every occurrence.

In [7]:
def token_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", str(text), flags=re.UNICODE))


def block_text_and_rects(block: dict):
    line_texts = []
    rects = []
    for line in block.get("lines", []):
        spans = [clean_text(span.get("text", "")) for span in line.get("spans", [])]
        line_text = clean_text(" ".join(x for x in spans if x))
        if line_text:
            line_texts.append(line_text)
            bbox = line.get("bbox")
            if bbox and len(bbox) == 4:
                rects.append([float(v) for v in bbox])
    text = clean_text(" ".join(line_texts))
    if not rects and block.get("bbox"):
        rects = [[float(v) for v in block["bbox"]]]
    return text, rects


def repeated_margin_keys(candidates: list[dict], page_count: int) -> set[str]:
    pages_by_key = defaultdict(set)
    for rec in candidates:
        if not rec["is_marginal"]:
            continue
        key = normalize_for_match(rec["text"])
        if not key or len(key) > 140:
            continue
        pages_by_key[key].add(rec["page_number"])

    minimum_pages = max(2, math.ceil(page_count * REPEATED_MARGIN_PAGE_FRACTION))
    return {key for key, pages in pages_by_key.items() if len(pages) >= minimum_pages}


def extract_pdf_paragraphs(pdf_path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    pdf_path = Path(pdf_path)
    doc_slug = safe_slug(pdf_path.stem)
    candidates = []
    page_quality = []

    with fitz.open(pdf_path) as doc:
        metadata_title = clean_text((doc.metadata or {}).get("title", ""))
        document_title = metadata_title or pdf_path.stem
        page_count = len(doc)

        for page_index, page in enumerate(doc, start=1):
            page_dict = page.get_text("dict", sort=True)
            page_height = float(page.rect.height)
            page_chars = 0
            block_position = 0

            for block in page_dict.get("blocks", []):
                if block.get("type", 0) != 0:
                    continue
                text, rects = block_text_and_rects(block)
                if not text:
                    continue

                page_chars += len(text)
                block_position += 1
                bbox = block.get("bbox") or rects[0]
                y0, y1 = float(bbox[1]), float(bbox[3])
                is_marginal = (
                    y1 <= page_height * MARGINAL_ZONE_FRACTION
                    or y0 >= page_height * (1.0 - MARGINAL_ZONE_FRACTION)
                )

                candidates.append({
                    "source_file": str(pdf_path),
                    "source_filename": pdf_path.name,
                    "doc_id": doc_slug,
                    "document_title": document_title,
                    "page_number": page_index,
                    "block_number": block_position,
                    "text": text,
                    "rects": rects,
                    "bbox": [float(v) for v in bbox],
                    "is_marginal": is_marginal,
                })

            page_quality.append({
                "source_file": str(pdf_path),
                "source_filename": pdf_path.name,
                "document_title": document_title,
                "page_number": page_index,
                "embedded_text_characters": page_chars,
                "needs_ocr_or_manual_review": page_chars < LOW_TEXT_PAGE_CHARS,
            })

    repeated_keys = repeated_margin_keys(candidates, page_count)
    kept = []
    paragraph_sequence = 0

    for rec in candidates:
        norm = normalize_for_match(rec["text"])
        if rec["is_marginal"] and norm in repeated_keys:
            continue
        if re.fullmatch(r"(?:page\s*)?\d+(?:\s+of\s+\d+)?", norm):
            continue
        if len(rec["text"]) < MIN_PARAGRAPH_CHARS:
            continue
        if token_count(rec["text"]) < MIN_PARAGRAPH_TOKENS:
            continue

        paragraph_sequence += 1
        text_hash = sha256_text(norm)
        kept.append({
            **{k: v for k, v in rec.items() if k not in {"rects", "bbox", "is_marginal"}},
            "paragraph_id": f"{rec['doc_id']}_p{rec['page_number']:04d}_b{rec['block_number']:03d}",
            "paragraph_order": paragraph_sequence,
            "token_count": token_count(rec["text"]),
            "text_hash": text_hash,
            "normalized_text": norm,
            "rects_json": json.dumps(rec["rects"]),
            "bbox_json": json.dumps(rec["bbox"]),
        })

    return pd.DataFrame(kept), pd.DataFrame(page_quality)


pdf_paths = sorted(INPUT_DIR.glob(PDF_PATTERN))
if not pdf_paths:
    raise FileNotFoundError(
        f"No PDFs matching {PDF_PATTERN!r} were found in {INPUT_DIR}. "
        "Upload at least one PDF and rerun this cell."
    )

paragraph_frames = []
quality_frames = []
for path in pdf_paths:
    paragraph_df, quality_df = extract_pdf_paragraphs(path)
    paragraph_frames.append(paragraph_df)
    quality_frames.append(quality_df)

paragraphs_df = pd.concat(paragraph_frames, ignore_index=True)
page_quality_df = pd.concat(quality_frames, ignore_index=True)

if paragraphs_df.empty:
    raise RuntimeError("No paragraph-like blocks passed the extraction filters.")

paragraphs_df["duplicate_occurrence_count"] = paragraphs_df.groupby("text_hash")["text_hash"].transform("size")
unique_paragraphs_df = paragraphs_df.drop_duplicates("text_hash", keep="first").reset_index(drop=True)
if MAX_UNIQUE_PARAGRAPHS is not None:
    unique_paragraphs_df = unique_paragraphs_df.head(MAX_UNIQUE_PARAGRAPHS).copy()

paragraphs_df.to_csv(OUTPUT_DIR / "extracted_paragraphs.csv", index=False)
page_quality_df.to_csv(OUTPUT_DIR / "page_extraction_quality.csv", index=False)

print("Documents:", paragraphs_df["source_filename"].nunique())
print("Paragraph occurrences:", len(paragraphs_df))
print("Unique paragraph texts to label:", len(unique_paragraphs_df))
print("Duplicate occurrences saved through caching:", len(paragraphs_df) - paragraphs_df["text_hash"].nunique())
print("Pages flagged for OCR/manual review:", int(page_quality_df["needs_ocr_or_manual_review"].sum()))
display(paragraphs_df[["paragraph_id", "source_filename", "page_number", "token_count", "text"]].head(10))

Documents: 1
Paragraph occurrences: 96
Unique paragraph texts to label: 91
Duplicate occurrences saved through caching: 5
Pages flagged for OCR/manual review: 0


,paragraph_id,source_filename,page_number,token_count,text
0,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b006,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,22,"way of exposing weaknesses in an administrative structure, and tragedy leads all to reevaluate their circumstances, be they organizations, communities, or individuals."
1,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b007,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,37,Michael McGuire is a professor in the School of Public and Environmental Affairs at Indiana University–Bloomington. His cur- rent research examines collaborative leader- ship i...
2,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b008,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,24,Debra Schneck is a doctoral student in the School of Public and Environmental Af- fairs at Indiana University–Bloomington. E-mail: dschneck@indiana.edu
3,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b013,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,103,What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? Th is essay argues that learning to manage actions supporting disaster ...
4,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b014,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,66,"Wise and fortunate indeed is that community that has so analyzed its problems and needs, and has so prepared to make use of catastrophe should it come by plans for carrying out..."
5,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b016,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,259,"What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? Or is the worst yet to come? (Kettl 2006). In the fol- lowing sections,..."
6,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b017,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,118,"cores of reports, newspaper accounts, docu- mentaries, and academic articles have been produced about the catastrophic U.S. hurri- cane of 2005, Hurricane Katrina. Many of thes..."
7,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b018,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,27,"Because much of the blame for the overall ineﬀ ective response to Katrina is placed on the Federal Emer- gency Management Agency (FEMA), as the nation’s"
8,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0002_b001,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,2,22,"practices” emanated from both governmental and nongovernmen- tal organizations, but rarely did one consistent set exist for a given area (Waugh 2007)."
9,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0002_b002,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,2,82,"Within two years after 9/11, FEMA had been absorbed into the newly cobbled together Department of Homeland Security along with more than 20 other organizational entities, many ..."


## 7. Select balanced few-shot examples and enforce leakage safeguards

Only OpenAI and Anthropic receive labeled examples. The Google strategy remains direct.

An example is excluded when:

- its normalized paragraph exactly matches an input paragraph;
- its maximum normalized-text similarity reaches the configured threshold;
- or its source-document name matches an input document and same-document exclusion is enabled.

In [8]:
def choose_balanced_examples(pool: pd.DataFrame, n_examples: int, seed: int) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    chosen_indices = []
    labels = [False, True]

    per_label = n_examples // 2
    for label in labels:
        idx = pool.index[pool["example_unlearning"].eq(label)].to_numpy()
        take = min(per_label, len(idx))
        if take:
            chosen_indices.extend(rng.choice(idx, size=take, replace=False).tolist())

    remaining = n_examples - len(chosen_indices)
    if remaining > 0:
        available = pool.index[~pool.index.isin(chosen_indices)].to_numpy()
        if len(available):
            chosen_indices.extend(rng.choice(available, size=min(remaining, len(available)), replace=False).tolist())

    return pool.loc[chosen_indices].copy().reset_index(drop=True)


def document_key(value: str) -> str:
    key = normalize_for_match(value)
    key = re.sub(r"\bpdf\b", " ", key)
    return re.sub(r"\s+", " ", key).strip()


def audit_and_filter_examples(candidates: pd.DataFrame, input_paragraphs: pd.DataFrame):
    input_norms = input_paragraphs["normalized_text"].tolist()
    input_hashes = set(input_paragraphs["text_hash"])
    input_document_keys = set()
    for value in input_paragraphs["source_filename"].tolist() + input_paragraphs["document_title"].tolist():
        key = document_key(value)
        if key:
            input_document_keys.add(key)

    audit_rows = []
    kept_rows = []

    for _, row in candidates.iterrows():
        norm = normalize_for_match(row["Text Content"])
        text_hash = sha256_text(norm)
        exact_match = text_hash in input_hashes

        max_similarity = 0.0
        if input_norms:
            max_similarity = max(rapid_ratio(norm, target) / 100.0 for target in input_norms)

        example_doc_key = document_key(row.get("Document", ""))
        same_document = bool(example_doc_key and example_doc_key in input_document_keys)

        reasons = []
        if exact_match:
            reasons.append("exact_text_overlap")
        if max_similarity >= LEAKAGE_SIMILARITY_THRESHOLD:
            reasons.append("near_text_overlap")
        if EXCLUDE_SAME_DOCUMENT_EXAMPLES and same_document:
            reasons.append("same_document")

        keep = not reasons
        audit_rows.append({
            "example_number": row.get("Number", ""),
            "example_unlearning": row["example_unlearning"],
            "example_document": row.get("Document", ""),
            "exact_match": exact_match,
            "max_similarity": max_similarity,
            "same_document": same_document,
            "kept": keep,
            "exclusion_reason": "; ".join(reasons),
            "text_preview": row["Text Content"][:240],
        })
        if keep:
            kept_rows.append(row.to_dict())

    return pd.DataFrame(kept_rows), pd.DataFrame(audit_rows)


# Audit the full pool first, then select a balanced set from safe examples.
safe_example_pool_df, example_leakage_audit_df = audit_and_filter_examples(
    examples_pool_df,
    unique_paragraphs_df,
)
prompt_examples_df = choose_balanced_examples(
    safe_example_pool_df,
    N_FEWSHOT_EXAMPLES,
    FEWSHOT_SEED,
)

example_leakage_audit_df.to_csv(OUTPUT_DIR / "example_leakage_audit.csv", index=False)
prompt_examples_df.to_csv(OUTPUT_DIR / "fewshot_examples_used.csv", index=False)

required_by_strategy = any(cfg["strategy"] == "definitions_examples_no_metadata" for cfg in MODEL_CONFIGS)
if required_by_strategy and len(prompt_examples_df) < MIN_SAFE_FEWSHOT_EXAMPLES:
    raise RuntimeError(
        f"Only {len(prompt_examples_df)} safe few-shot examples remain after leakage checks. "
        f"At least {MIN_SAFE_FEWSHOT_EXAMPLES} are required. Add approved examples from other documents."
    )

print("Candidate examples:", len(examples_pool_df))
print("Safe examples after document/text leakage checks:", len(safe_example_pool_df))
print("Examples used in OpenAI/Anthropic prompts:", len(prompt_examples_df))
print("Selected label counts:", prompt_examples_df["example_unlearning"].value_counts().to_dict())
display(example_leakage_audit_df[~example_leakage_audit_df["kept"]].head(20))

Candidate examples: 49
Safe examples after document/text leakage checks: 49
Examples used in OpenAI/Anthropic prompts: 6
Selected label counts: {False: 3, True: 3}


,example_number,example_unlearning,example_document,exact_match,max_similarity,same_document,kept,exclusion_reason,text_preview


## 8. Shared structured output and provider-specific prompts

The output schema remains aligned with the earlier experiments: binary unlearning, one target type, agency, confidence, and rationale.

In [9]:
TARGET_TYPES = [
    "Leadership",
    "laws_plans_policies",
    "capabilities",
    "funds_resources",
    "misc_organizational",
    "none",
]

TARGET_DISPLAY = {
    "Leadership": "Leadership",
    "laws_plans_policies": "Laws, plans and policies",
    "capabilities": "Capabilities",
    "funds_resources": "Funds and resources",
    "misc_organizational": "Miscellaneous organizational",
    "none": "None",
    "needs_review": "Needs review",
}


class AnnotationOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    unlearning_present: bool
    target_type: Literal[
        "Leadership",
        "laws_plans_policies",
        "capabilities",
        "funds_resources",
        "misc_organizational",
        "none",
    ]
    agency: Optional[str] = None
    confidence: float = Field(ge=0.0, le=1.0)
    rationale: str


ANNOTATION_JSON_SCHEMA = AnnotationOutput.model_json_schema()
BASE_SYSTEM_PROMPT = "Return only the requested structured output. Do not include markdown."

COMMON_TASK_BLOCK = """You are coding federal disaster-policy text for organizational unlearning.

Task:
1. Decide whether the target paragraph contains unlearning.
2. If yes, assign exactly one target type.
3. Identify the agency or sub-unit only when stated or clearly identifiable in the paragraph.
4. Give one concise rationale grounded in the paragraph.

Unlearning target types:
- Leadership
- laws_plans_policies
- capabilities
- funds_resources
- misc_organizational
- none

When unlearning_present is false, target_type must be none.""".strip()


def example_codes(row: pd.Series) -> str:
    codes = clean_text(row.get("Codes", ""))
    if codes:
        return codes

    parts = [f"Unlearning: {'Yes' if row['example_unlearning'] else 'No'}"]
    target = clean_text(row.get("Target", ""))
    agency = clean_text(row.get("Government Agency", ""))
    if target:
        parts.append(f"Target: {target}")
    if agency:
        parts.append(f"Government Agency: {agency}")
    return " | ".join(parts)


def serialize_examples_no_metadata(df: pd.DataFrame) -> str:
    blocks = []
    for i, (_, row) in enumerate(df.iterrows(), start=1):
        blocks.append("\n".join([
            f"Example {i}",
            "Text Content:",
            row["Text Content"],
            "Human labels:",
            f"Unlearning: {'Yes' if row['example_unlearning'] else 'No'}",
            f"Codes: {example_codes(row)}",
        ]))
    return "\n\n---\n\n".join(blocks)


EXAMPLES_TEXT_NO_METADATA = serialize_examples_no_metadata(prompt_examples_df)


def build_user_prompt(strategy: str, row: pd.Series) -> str:
    paragraph_id = row.get("paragraph_id", row.get("text_hash", "paragraph"))
    paragraph = row["text"]

    if strategy == "direct_no_context":
        return f"""{COMMON_TASK_BLOCK}

Paragraph ID: {paragraph_id}
Paragraph:
{paragraph}""".strip()

    if strategy == "definitions_examples_no_metadata":
        return f"""{COMMON_TASK_BLOCK}

Codebook:
{CODEBOOK_TEXT}

Labeled examples:
{EXAMPLES_TEXT_NO_METADATA}

Paragraph ID: {paragraph_id}
Paragraph:
{paragraph}""".strip()

    raise ValueError(f"Unsupported strategy: {strategy}")


def build_messages(cfg: dict, row: pd.Series) -> list[dict]:
    return [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(cfg["strategy"], row)},
    ]


def prompt_hash(cfg: dict, row: pd.Series) -> str:
    payload = {
        "prompt_version": PROMPT_VERSION,
        "provider": cfg["provider"],
        "model": cfg["model"],
        "strategy": cfg["strategy"],
        "messages": build_messages(cfg, row),
        "schema": ANNOTATION_JSON_SCHEMA,
    }
    return sha256_text(json.dumps(payload, ensure_ascii=False, sort_keys=True))


preview = unique_paragraphs_df.iloc[0]
for cfg in MODEL_CONFIGS:
    prompt = build_messages(cfg, preview)[1]["content"]
    print("\n", cfg["provider"], cfg["strategy"], "characters:", len(prompt))
    print(prompt[:700], "...")


 openai definitions_examples_no_metadata characters: 18024
You are coding federal disaster-policy text for organizational unlearning.

Task:
1. Decide whether the target paragraph contains unlearning.
2. If yes, assign exactly one target type.
3. Identify the agency or sub-unit only when stated or clearly identifiable in the paragraph.
4. Give one concise rationale grounded in the paragraph.

Unlearning target types:
- Leadership
- laws_plans_policies
- capabilities
- funds_resources
- misc_organizational
- none

When unlearning_present is false, target_type must be none.

Codebook:
Code: Step 1 -- Binary classification based on unlearning definition
Definition: 
Detection logic: 
Examples: 
Positive clarification: 
Negative clarification: 

Code: U ...

 anthropic definitions_examples_no_metadata characters: 18024
You are coding federal disaster-policy text for organizational unlearning.

Task:
1. Decide whether the target paragraph contains unlearning.
2. If yes, assign exactly one 

## 9. Provider API calls, retry handling, and resumable JSONL logs

Each provider writes to a separate JSONL file. A completed paragraph is skipped on rerun only when the model, strategy, prompt version, schema, and paragraph text all match.

In [10]:
class NonRetryableRequestError(RuntimeError):
    pass


def retry_call(fn, *args, max_attempts=MAX_RETRY_ATTEMPTS, **kwargs):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(*args, **kwargs)
        except NonRetryableRequestError:
            raise
        except Exception as exc:
            last_error = exc
            message = str(exc).lower()
            nonretryable_markers = [
                "invalid_request_error", "error code: 400", "400 invalid_argument",
                "unknown name", "deprecated for this model", "cannot find field",
            ]
            if any(marker in message for marker in nonretryable_markers):
                raise NonRetryableRequestError(str(exc)) from exc
            if attempt == max_attempts:
                raise
            wait = min(60.0, 4.0 * (2 ** (attempt - 1)))
            print(f"Transient error ({attempt}/{max_attempts}): {exc}\nRetrying in {wait:.1f}s...")
            time.sleep(wait)
    raise last_error


def usage_result(raw_response, input_tokens=0, output_tokens=0, total_tokens=0, reasoning_tokens=0, latency=0.0):
    return {
        "raw_response": raw_response,
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or (input_tokens or 0) + (output_tokens or 0)),
        "reasoning_tokens": int(reasoning_tokens or 0),
        "latency_seconds": float(latency),
    }


def call_openai(messages: list[dict], cfg: dict) -> dict:
    from openai import OpenAI

    client = OpenAI(api_key=os.environ[cfg["api_key_env"]])
    system_text = next(m["content"] for m in messages if m["role"] == "system")
    response_input = [m for m in messages if m["role"] != "system"]

    start = time.time()
    response = client.responses.parse(
        model=cfg["model"],
        instructions=system_text,
        input=response_input,
        reasoning={"effort": cfg.get("reasoning_effort", "low")},
        max_output_tokens=MAX_OUTPUT_TOKENS,
        text_format=AnnotationOutput,
        store=False,
    )
    latency = time.time() - start

    parsed = response.output_parsed
    if parsed is None:
        raise RuntimeError(f"OpenAI returned no parsed output: {response.output_text!r}")

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", 0) or 0
    output_tokens = getattr(usage, "output_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or input_tokens + output_tokens
    details = getattr(usage, "output_tokens_details", None)
    reasoning_tokens = getattr(details, "reasoning_tokens", 0) or 0

    return usage_result(
        json.dumps(parsed.model_dump(), ensure_ascii=False),
        input_tokens, output_tokens, total_tokens, reasoning_tokens, latency,
    )


def call_anthropic(messages: list[dict], cfg: dict) -> dict:
    from anthropic import Anthropic

    client = Anthropic(api_key=os.environ[cfg["api_key_env"]])
    system_text = "\n".join(m["content"] for m in messages if m["role"] == "system")
    anthropic_messages = [m for m in messages if m["role"] in {"user", "assistant"}]

    request = {
        "model": cfg["model"],
        "max_tokens": MAX_OUTPUT_TOKENS,
        "system": system_text,
        "messages": anthropic_messages,
        "output_format": AnnotationOutput,
    }
    if cfg.get("effort") is not None:
        request["output_config"] = {"effort": cfg["effort"]}

    start = time.time()
    response = client.messages.parse(**request)
    latency = time.time() - start

    parsed = response.parsed_output
    if parsed is None:
        raise RuntimeError("Anthropic returned no parsed output.")

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", 0) or 0
    output_tokens = getattr(usage, "output_tokens", 0) or 0

    return usage_result(
        json.dumps(parsed.model_dump(), ensure_ascii=False),
        input_tokens, output_tokens, input_tokens + output_tokens, 0, latency,
    )


def call_gemini(messages: list[dict], cfg: dict) -> dict:
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=os.environ[cfg["api_key_env"]])
    system_text = next(m["content"] for m in messages if m["role"] == "system")
    user_text = "\n\n".join(m["content"] for m in messages if m["role"] == "user")

    start = time.time()
    response = client.models.generate_content(
        model=cfg["model"],
        contents=user_text,
        config=types.GenerateContentConfig(
            system_instruction=system_text,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            response_mime_type="application/json",
            response_json_schema=ANNOTATION_JSON_SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level=cfg.get("thinking_level", "minimal")
            ),
        ),
    )
    latency = time.time() - start

    raw = response.text or ""
    try:
        parsed = AnnotationOutput.model_validate_json(raw)
    except ValidationError as exc:
        raise RuntimeError(f"Gemini structured output failed validation: {exc}; raw={raw!r}") from exc

    usage = getattr(response, "usage_metadata", None)
    input_tokens = getattr(usage, "prompt_token_count", 0) or 0
    output_tokens = getattr(usage, "candidates_token_count", 0) or 0
    total_tokens = getattr(usage, "total_token_count", 0) or input_tokens + output_tokens
    thinking_tokens = getattr(usage, "thoughts_token_count", 0) or 0

    return usage_result(
        json.dumps(parsed.model_dump(), ensure_ascii=False),
        input_tokens, output_tokens, total_tokens, thinking_tokens, latency,
    )


def mock_call(messages: list[dict], cfg: dict) -> dict:
    """Deterministic smoke-test output. Never use as research data."""
    text = messages[-1]["content"].rsplit("Paragraph:\n", 1)[-1].lower()
    positive_terms = ["replace", "abandon", "move away", "eliminate", "reorganize", "reform"]
    present = any(term in text for term in positive_terms)
    target = "laws_plans_policies" if present else "none"
    parsed = AnnotationOutput(
        unlearning_present=present,
        target_type=target,
        agency=None,
        confidence=0.75,
        rationale="Mock-mode deterministic classification for pipeline testing only.",
    )
    return usage_result(json.dumps(parsed.model_dump()), latency=0.001)


def call_provider(messages: list[dict], cfg: dict) -> dict:
    if MOCK_MODE:
        return mock_call(messages, cfg)
    if cfg["provider"] == "openai":
        return retry_call(call_openai, messages, cfg)
    if cfg["provider"] == "anthropic":
        return retry_call(call_anthropic, messages, cfg)
    if cfg["provider"] == "gemini":
        return retry_call(call_gemini, messages, cfg)
    raise ValueError(f"Unsupported provider: {cfg['provider']}")


def provider_jsonl_path(cfg: dict) -> Path:
    return RAW_DIR / f"{cfg['provider']}__{safe_slug(cfg['model'])}__{PROMPT_VERSION}.jsonl"


def load_success_records(path: Path) -> dict[str, dict]:
    records = {}
    if not path.exists() or not RESUME_FROM_JSONL:
        return records
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            rec = json.loads(line)
            if rec.get("status") == "ok" and rec.get("run_key"):
                records[rec["run_key"]] = rec
    return records


def append_jsonl(path: Path, record: dict):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def semantic_validate(parsed: AnnotationOutput) -> list[str]:
    warnings_list = []
    if not parsed.unlearning_present and parsed.target_type != "none":
        warnings_list.append("negative_prediction_with_non_none_target")
    if parsed.unlearning_present and parsed.target_type == "none":
        warnings_list.append("positive_prediction_with_none_target")
    return warnings_list


def run_provider(cfg: dict, rows: pd.DataFrame) -> pd.DataFrame:
    path = provider_jsonl_path(cfg)
    prior = load_success_records(path)
    results = []

    for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f"{cfg['provider']} labeling"):
        p_hash = prompt_hash(cfg, row)
        run_key = sha256_text("|".join([
            PROMPT_VERSION, cfg["provider"], cfg["model"], cfg["strategy"],
            row["text_hash"], p_hash,
        ]))

        if run_key in prior:
            results.append(prior[run_key])
            continue

        base = {
            "run_key": run_key,
            "timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "prompt_version": PROMPT_VERSION,
            "prompt_sha256": p_hash,
            "provider": cfg["provider"],
            "model": cfg["model"],
            "strategy": cfg["strategy"],
            "text_hash": row["text_hash"],
            "representative_paragraph_id": row["paragraph_id"],
            "representative_document": row["source_filename"],
            "representative_page": int(row["page_number"]),
            "text": row["text"],
        }

        if not RUN_API_CALLS and not MOCK_MODE:
            record = {
                **base,
                "status": "dry_run",
                "pred_unlearning": None,
                "pred_target_type": None,
                "pred_agency": None,
                "confidence": None,
                "rationale": None,
                "semantic_warning": None,
                "raw_response": None,
                "input_tokens": 0,
                "output_tokens": 0,
                "reasoning_tokens": 0,
                "total_tokens": 0,
                "latency_seconds": 0.0,
                "error": None,
            }
        else:
            try:
                api_result = call_provider(build_messages(cfg, row), cfg)
                parsed = AnnotationOutput.model_validate_json(api_result["raw_response"])
                semantic_warnings = semantic_validate(parsed)
                target = parsed.target_type if parsed.unlearning_present else "none"

                record = {
                    **base,
                    "status": "ok",
                    "pred_unlearning": bool(parsed.unlearning_present),
                    "pred_target_type": target,
                    "pred_agency": clean_text(parsed.agency),
                    "confidence": float(parsed.confidence),
                    "rationale": clean_text(parsed.rationale),
                    "semantic_warning": "; ".join(semantic_warnings),
                    **api_result,
                    "error": None,
                }
            except Exception as exc:
                record = {
                    **base,
                    "status": "error",
                    "pred_unlearning": None,
                    "pred_target_type": None,
                    "pred_agency": None,
                    "confidence": None,
                    "rationale": None,
                    "semantic_warning": None,
                    "raw_response": None,
                    "input_tokens": 0,
                    "output_tokens": 0,
                    "reasoning_tokens": 0,
                    "total_tokens": 0,
                    "latency_seconds": 0.0,
                    "error": repr(exc),
                }

        append_jsonl(path, record)
        results.append(record)
        time.sleep(REQUEST_SLEEP_SECONDS)

    result_df = pd.DataFrame(results)
    result_df.to_csv(OUTPUT_DIR / f"{cfg['provider']}_predictions_snapshot.csv", index=False)
    return result_df

print("Provider functions loaded.")

Provider functions loaded.


## 10. Run the three providers

This labels each unique paragraph once per provider. Duplicate paragraph occurrences inherit the cached prediction later.

In [15]:
provider_result_frames = []
for cfg in MODEL_CONFIGS:
    frame = run_provider(cfg, unique_paragraphs_df)
    provider_result_frames.append(frame)

model_predictions_unique_df = pd.concat(provider_result_frames, ignore_index=True)
model_predictions_unique_df.to_csv(OUTPUT_DIR / "model_predictions_unique_texts.csv", index=False)

status_summary = model_predictions_unique_df.groupby(["provider", "status"]).size().unstack(fill_value=0)
display(status_summary)

if not MOCK_MODE and RUN_API_CALLS:
    errors = model_predictions_unique_df[model_predictions_unique_df["status"].ne("ok")]
    if not errors.empty:
        display(errors[["provider", "representative_paragraph_id", "error"]].head(20))
        raise RuntimeError(
            f"{len(errors)} provider calls did not complete successfully. Fix them and rerun; completed calls will resume."
        )

if not RUN_API_CALLS and not MOCK_MODE:
    raise RuntimeError(
        "Dry run completed, but no labels were generated. Set UNLEARNING_RUN_API_CALLS=1 or RUN_API_CALLS=True and rerun from the API-key cell."
    )

openai labeling:   0%|          | 0/91 [00:00<?, ?it/s]

anthropic labeling:   0%|          | 0/91 [00:00<?, ?it/s]

gemini labeling:   0%|          | 0/91 [00:00<?, ?it/s]

status,ok
provider,
anthropic,91
gemini,91
openai,91


## 11. Expand cached predictions back to every paragraph occurrence

In [16]:
valid_unique_predictions_df = model_predictions_unique_df[
    model_predictions_unique_df["status"].eq("ok")
].copy()

expected_providers = {cfg["provider"] for cfg in MODEL_CONFIGS}
observed_providers = set(valid_unique_predictions_df["provider"].unique())
if observed_providers != expected_providers:
    raise RuntimeError(f"Expected providers {expected_providers}, observed {observed_providers}.")

counts_per_hash = valid_unique_predictions_df.groupby("text_hash")["provider"].nunique()
if not counts_per_hash.eq(len(expected_providers)).all():
    bad_hashes = counts_per_hash[counts_per_hash.ne(len(expected_providers))]
    raise RuntimeError(f"Some paragraph texts are missing provider labels: {bad_hashes.head().to_dict()}")

model_predictions_long_df = paragraphs_df.merge(
    valid_unique_predictions_df.drop(columns=["text", "representative_paragraph_id", "representative_document", "representative_page"]),
    on="text_hash",
    how="left",
    validate="many_to_many",
)
model_predictions_long_df.to_csv(OUTPUT_DIR / "model_predictions_long.csv", index=False)

wide_parts = []
for provider in sorted(expected_providers):
    part = valid_unique_predictions_df[valid_unique_predictions_df["provider"].eq(provider)].copy()
    part = part[[
        "text_hash", "model", "strategy", "pred_unlearning", "pred_target_type",
        "pred_agency", "confidence", "rationale", "semantic_warning",
    ]]
    part = part.rename(columns={
        col: f"{provider}_{col}" for col in part.columns if col != "text_hash"
    })
    wide_parts.append(part)

model_predictions_wide_unique_df = wide_parts[0]
for part in wide_parts[1:]:
    model_predictions_wide_unique_df = model_predictions_wide_unique_df.merge(
        part, on="text_hash", how="inner", validate="one_to_one"
    )

model_predictions_wide_df = paragraphs_df.merge(
    model_predictions_wide_unique_df,
    on="text_hash",
    how="left",
    validate="many_to_one",
)
model_predictions_wide_df.to_csv(OUTPUT_DIR / "model_predictions_wide.csv", index=False)

print("Long prediction rows:", len(model_predictions_long_df))
print("Wide paragraph-occurrence rows:", len(model_predictions_wide_df))
display(model_predictions_wide_df.head())

Long prediction rows: 288
Wide paragraph-occurrence rows: 96


,source_file,source_filename,doc_id,document_title,page_number,block_number,text,paragraph_id,paragraph_order,token_count,text_hash,normalized_text,rects_json,bbox_json,duplicate_occurrence_count,anthropic_model,anthropic_strategy,anthropic_pred_unlearning,anthropic_pred_target_type,anthropic_pred_agency,anthropic_confidence,anthropic_rationale,anthropic_semantic_warning,gemini_model,gemini_strategy,gemini_pred_unlearning,gemini_pred_target_type,gemini_pred_agency,gemini_confidence,gemini_rationale,gemini_semantic_warning,openai_model,openai_strategy,openai_pred_unlearning,openai_pred_target_type,openai_pred_agency,openai_confidence,openai_rationale,openai_semantic_warning
0,/content/unlearning_pipeline/input_documents/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi,What if Hurricane Katrina Hit in 2020? The Need for Strategic Management of Disasters,1,6,"way of exposing weaknesses in an administrative structure, and tragedy leads all to reevaluate their circumstances, be they organizations, communities, or individuals.",Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b006,1,22,bcf9f758724a07bd5d41c01901a76d37f1094c8f87a8a8cb36bdcde8b53ed38b,way of exposing weaknesses in an administrative structure and tragedy leads all to reevaluate their circumstances be they organizations communities or individuals,"[[253.76651000976562, 223.01376342773438, 444.7665710449219, 233.01376342773438], [253.76651000976562, 235.01376342773438, 448.0864562988281, 245.01376342773438], [253.76651000...","[253.76651000976562, 223.01376342773438, 457.48651123046875, 269.0137634277344]",1,claude-sonnet-5,definitions_examples_no_metadata,False,none,,0.85,"The paragraph offers general reflection on how tragedy exposes weaknesses and prompts reevaluation, but does not name a specific prior policy, practice, or system nor call for ...",,gemini-3.1-flash-lite,direct_no_context,True,misc_organizational,,0.9,"The text describes an organizational process of reevaluating circumstances following exposure of weaknesses, indicating a shift in collective knowledge or structure.",,gpt-5.6-terra,definitions_examples_no_metadata,False,none,,0.98,The paragraph generally notes that tragedy prompts reevaluation but identifies no specific prior practice or structure to abandon or replace.,
1,/content/unlearning_pipeline/input_documents/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi,What if Hurricane Katrina Hit in 2020? The Need for Strategic Management of Disasters,1,7,Michael McGuire is a professor in the School of Public and Environmental Affairs at Indiana University–Bloomington. His cur- rent research examines collaborative leader- ship i...,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b007,2,37,342b3d560f4a7cc64b1cd6c96c2cac84a6769511344c26a844ebcfb032ca4241,michael mcguire is a professor in the school of public and environmental affairs at indiana university bloomington his cur rent research examines collaborative leader ship in t...,"[[475.3699951171875, 225.7054443359375, 563.3314208984375, 231.7054443359375], [475.3699951171875, 234.7054443359375, 568.9403076171875, 240.7054443359375], [475.3699951171875,...","[475.3699951171875, 225.7054443359375, 571.006103515625, 276.7054443359375]",1,claude-sonnet-5,definitions_examples_no_metadata,False,none,,0.98,This paragraph is merely an author biography and contains no discussion of policy failure or calls for chang

## 12. Inter-coder reliability

The notebook reports:

- pairwise percent agreement and Cohen's kappa;
- overall Fleiss' kappa for three coders;
- nominal Krippendorff's alpha;
- binary unlearning reliability and target-label reliability;
- both unique-text and all-occurrence versions, so repeated text does not silently inflate reliability.

In [17]:
PROVIDER_ORDER = [cfg["provider"] for cfg in MODEL_CONFIGS]


def krippendorff_alpha_nominal(ratings: pd.DataFrame) -> float:
    """Nominal alpha for a units x coders matrix, allowing missing ratings."""
    array = ratings.to_numpy(dtype=object)
    categories = sorted({x for x in array.ravel() if pd.notna(x)}, key=str)
    if not categories:
        return np.nan

    observed_num = 0.0
    observed_den = 0.0
    total_counts = Counter()

    for row in array:
        values = [x for x in row if pd.notna(x)]
        n = len(values)
        if n < 2:
            continue
        counts = Counter(values)
        observed_num += sum(count * (n - count) for count in counts.values())
        observed_den += n * (n - 1)
        total_counts.update(values)

    if observed_den == 0:
        return np.nan
    observed_disagreement = observed_num / observed_den

    total_n = sum(total_counts.values())
    if total_n < 2:
        return np.nan
    expected_num = sum(count * (total_n - count) for count in total_counts.values())
    expected_den = total_n * (total_n - 1)
    expected_disagreement = expected_num / expected_den

    if expected_disagreement == 0:
        return 1.0 if observed_disagreement == 0 else np.nan
    return 1.0 - observed_disagreement / expected_disagreement


def fleiss_from_ratings(ratings: pd.DataFrame) -> float:
    categories = sorted(pd.unique(ratings.to_numpy().ravel()).tolist(), key=str)
    categories = [c for c in categories if pd.notna(c)]
    complete = ratings.dropna()
    if complete.empty or not categories:
        return np.nan

    count_matrix = np.array([
        [(row == category).sum() for category in categories]
        for row in complete.to_numpy(dtype=object)
    ], dtype=float)

    try:
        return float(fleiss_kappa(count_matrix, method="fleiss"))
    except Exception:
        return np.nan


def pairwise_reliability(ratings: pd.DataFrame, dimension: str, scope: str) -> pd.DataFrame:
    rows = []
    for coder_a, coder_b in combinations(ratings.columns, 2):
        pair = ratings[[coder_a, coder_b]].dropna()
        if pair.empty:
            continue
        agreement = float((pair[coder_a] == pair[coder_b]).mean())
        try:
            kappa = float(cohen_kappa_score(pair[coder_a], pair[coder_b]))
        except Exception:
            kappa = np.nan
        rows.append({
            "dimension": dimension,
            "scope": scope,
            "coder_a": coder_a,
            "coder_b": coder_b,
            "n_units": len(pair),
            "percent_agreement": agreement,
            "cohen_kappa": kappa,
        })
    return pd.DataFrame(rows)


def overall_reliability(ratings: pd.DataFrame, dimension: str, scope: str) -> dict:
    complete = ratings.dropna()
    return {
        "dimension": dimension,
        "scope": scope,
        "n_units_total": len(ratings),
        "n_units_complete": len(complete),
        "n_coders": len(ratings.columns),
        "unanimous_agreement": float(complete.nunique(axis=1).eq(1).mean()) if len(complete) else np.nan,
        "fleiss_kappa": fleiss_from_ratings(ratings),
        "krippendorff_alpha_nominal": krippendorff_alpha_nominal(ratings),
    }


def ratings_matrix(source: pd.DataFrame, value_column: str, index_column: str) -> pd.DataFrame:
    return source.pivot_table(
        index=index_column,
        columns="provider",
        values=value_column,
        aggfunc="first",
    ).reindex(columns=PROVIDER_ORDER)


reliability_pairwise_frames = []
reliability_overall_rows = []

for scope, source, index_col in [
    ("unique_text", valid_unique_predictions_df, "text_hash"),
    ("all_occurrences", model_predictions_long_df, "paragraph_id"),
]:
    binary = ratings_matrix(source, "pred_unlearning", index_col)
    target = ratings_matrix(source, "pred_target_type", index_col)

    reliability_pairwise_frames.append(pairwise_reliability(binary, "unlearning_binary", scope))
    reliability_pairwise_frames.append(pairwise_reliability(target, "target_including_none", scope))
    reliability_overall_rows.append(overall_reliability(binary, "unlearning_binary", scope))
    reliability_overall_rows.append(overall_reliability(target, "target_including_none", scope))

    any_positive = binary.eq(True).any(axis=1)
    target_any_positive = target.loc[any_positive]
    reliability_pairwise_frames.append(
        pairwise_reliability(target_any_positive, "target_on_any_model_positive", scope)
    )
    reliability_overall_rows.append(
        overall_reliability(target_any_positive, "target_on_any_model_positive", scope)
    )

reliability_pairwise_df = pd.concat(reliability_pairwise_frames, ignore_index=True)
reliability_overall_df = pd.DataFrame(reliability_overall_rows)

# Per-document binary reliability on paragraph occurrences.
per_document_rows = []
for document, subset in model_predictions_long_df.groupby("source_filename"):
    matrix = ratings_matrix(subset, "pred_unlearning", "paragraph_id")
    rec = overall_reliability(matrix, "unlearning_binary", "document_occurrences")
    rec["source_filename"] = document
    per_document_rows.append(rec)
reliability_by_document_df = pd.DataFrame(per_document_rows)

reliability_pairwise_df.to_csv(OUTPUT_DIR / "reliability_pairwise.csv", index=False)
reliability_overall_df.to_csv(OUTPUT_DIR / "reliability_overall.csv", index=False)
reliability_by_document_df.to_csv(OUTPUT_DIR / "reliability_by_document.csv", index=False)

display(reliability_overall_df)
display(reliability_pairwise_df)

,dimension,scope,n_units_total,n_units_complete,n_coders,unanimous_agreement,fleiss_kappa,krippendorff_alpha_nominal
0,unlearning_binary,unique_text,91,91,3,0.846154,0.490536,0.492402
1,target_including_none,unique_text,91,91,3,0.846154,0.488764,0.490637
2,target_on_any_model_positive,unique_text,18,18,3,0.222222,0.179245,0.194444
3,unlearning_binary,all_occurrences,96,96,3,0.854167,0.493912,0.495670
4,target_including_none,all_occurrences,96,96,3,0.854167,0.491228,0.492995
5,target_on_any_model_positive,all_occurrences,18,18,3,0.222222,0.179245,0.194444


,dimension,scope,coder_a,coder_b,n_units,percent_agreement,cohen_kappa
0,unlearning_binary,unique_text,openai,anthropic,91,0.923077,0.491620
1,unlearning_binary,unique_text,openai,gemini,91,0.879121,0.464419
2,unlearning_binary,unique_text,anthropic,gemini,91,0.890110,0.528008
3,target_including_none,unique_text,openai,anthropic,91,0.923077,0.501955
4,target_including_none,unique_text,openai,gemini,91,0.868132,0.433904
5,target_including_none,unique_text,anthropic,gemini,91,0.890110,0.542943
6,target_on_any_model_positive,unique_text,openai,anthropic,18,0.611111,0.315217
7,target_on_any_model_positive,unique_text,openai,gemini,18,0.333333,0.136000
8,target_on_any_model_positive,unique_text,anthropic,gemini,18,0.444444,0.246862
9,unlearning_binary,all_occurrences,openai,anthropic,96,0.927083,0.493976


## 13. Consensus labels and review priority

Binary consensus is a two-of-three majority. Target consensus is assigned only when the binary consensus is Yes. If the positive models disagree on the target and no target has two votes, the target is marked `needs_review` rather than forced.

In [18]:
def consensus_for_text_hash(group: pd.DataFrame) -> dict:
    group = group.set_index("provider").reindex(PROVIDER_ORDER).reset_index()
    yes_votes = int(group["pred_unlearning"].eq(True).sum())
    no_votes = int(group["pred_unlearning"].eq(False).sum())
    consensus_yes = yes_votes >= 2

    if yes_votes == 3:
        agreement_bucket = "unanimous_yes"
    elif yes_votes == 2:
        agreement_bucket = "majority_yes"
    elif yes_votes == 1:
        agreement_bucket = "majority_no_one_positive"
    else:
        agreement_bucket = "unanimous_no"

    positive_group = group[group["pred_unlearning"].eq(True)]
    target_consensus = "none"
    target_vote_summary = ""
    target_agreement = True

    if consensus_yes:
        counts = Counter(positive_group["pred_target_type"].dropna().tolist())
        target_vote_summary = "; ".join(f"{k}:{v}" for k, v in counts.most_common())
        if counts:
            top_target, top_count = counts.most_common(1)[0]
            if top_count >= 2 or len(counts) == 1:
                target_consensus = top_target
            else:
                target_consensus = "needs_review"
                target_agreement = False
        else:
            target_consensus = "needs_review"
            target_agreement = False

    agreeing = group[group["pred_unlearning"].eq(consensus_yes)]
    consensus_confidence = float(agreeing["confidence"].mean()) if not agreeing.empty else np.nan

    agencies = sorted({
        clean_text(value)
        for value in positive_group["pred_agency"].tolist()
        if clean_text(value)
    })

    rationale_source = agreeing.sort_values("confidence", ascending=False).head(1)
    consensus_rationale = clean_text(rationale_source["rationale"].iloc[0]) if not rationale_source.empty else ""

    if agreement_bucket in {"majority_yes", "majority_no_one_positive"} or not target_agreement:
        review_priority = "High"
    elif agreement_bucket == "unanimous_yes":
        review_priority = "Medium"
    else:
        review_priority = "Low"

    result = {
        "text_hash": group["text_hash"].iloc[0],
        "yes_votes": yes_votes,
        "no_votes": no_votes,
        "consensus_unlearning": consensus_yes,
        "agreement_bucket": agreement_bucket,
        "consensus_target_type": target_consensus,
        "consensus_target_display": TARGET_DISPLAY.get(target_consensus, target_consensus),
        "target_vote_summary": target_vote_summary,
        "consensus_agencies": "; ".join(agencies),
        "consensus_confidence": consensus_confidence,
        "consensus_rationale": consensus_rationale,
        "review_priority": review_priority,
    }

    for _, row in group.iterrows():
        prefix = row["provider"]
        result[f"{prefix}_model"] = row["model"]
        result[f"{prefix}_strategy"] = row["strategy"]
        result[f"{prefix}_unlearning"] = row["pred_unlearning"]
        result[f"{prefix}_target_type"] = row["pred_target_type"]
        result[f"{prefix}_agency"] = row["pred_agency"]
        result[f"{prefix}_confidence"] = row["confidence"]
        result[f"{prefix}_rationale"] = row["rationale"]

    return result


consensus_unique_df = pd.DataFrame([
    consensus_for_text_hash(group)
    for _, group in valid_unique_predictions_df.groupby("text_hash", sort=False)
])

consensus_annotations_df = paragraphs_df.merge(
    consensus_unique_df,
    on="text_hash",
    how="left",
    validate="many_to_one",
)
consensus_annotations_df.to_csv(OUTPUT_DIR / "consensus_annotations.csv", index=False)

print(consensus_annotations_df["agreement_bucket"].value_counts())
display(consensus_annotations_df[[
    "paragraph_id", "source_filename", "page_number", "agreement_bucket",
    "consensus_unlearning", "consensus_target_display", "review_priority", "text",
]].head(20))

agreement_bucket
unanimous_no                78
majority_no_one_positive     9
majority_yes                 5
unanimous_yes                4
Name: count, dtype: int64


,paragraph_id,source_filename,page_number,agreement_bucket,consensus_unlearning,consensus_target_display,review_priority,text
0,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b006,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,majority_no_one_positive,False,None,High,"way of exposing weaknesses in an administrative structure, and tragedy leads all to reevaluate their circumstances, be they organizations, communities, or individuals."
1,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b007,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,unanimous_no,False,None,Low,Michael McGuire is a professor in the School of Public and Environmental Affairs at Indiana University–Bloomington. His cur- rent research examines collaborative leader- ship i...
2,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b008,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,unanimous_no,False,None,Low,Debra Schneck is a doctoral student in the School of Public and Environmental Af- fairs at Indiana University–Bloomington. E-mail: dschneck@indiana.edu
3,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b013,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,majority_no_one_positive,False,None,High,What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? Th is essay argues that learning to manage actions supporting disaster ...
4,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b014,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,unanimous_no,False,None,Low,"Wise and fortunate indeed is that community that has so analyzed its problems and needs, and has so prepared to make use of catastrophe should it come by plans for carrying out..."
5,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b016,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,unanimous_no,False,None,Low,"What would another Hurricane Katrina bring in 2020? Is government at all levels ready for such an event? Or is the worst yet to come? (Kettl 2006). In the fol- lowing sections,..."
6,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b017,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,unanimous_no,False,None,Low,"cores of reports, newspaper accounts, docu- mentaries, and academic articles have been produced about the catastrophic U.S. hurri- cane of 2005, Hurricane Katrina. Many of thes..."
7,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0001_b018,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,1,unanimous_no,False,None,Low,"Because much of the blame for the overall ineﬀ ective response to Katrina is placed on the Federal Emer- gency Management Agency (FEMA), as the nation’s"
8,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0002_b001,Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,2,unanimous_no,False,None,Low,"practices” emanated from both governmental and nongovernmen- tal organizations, but rarely did one consistent set exist for a given area (Waugh 2007)."
9,Public_Administration_Review_2010_McGuire_What_if_Hurricane_Katrina_Hi_p0002_b002,Public Administration Review - 2010 - McGuire - What if Hur

## 14. Export one review workbook

In [19]:
RESULTS_WORKBOOK_PATH = OUTPUT_DIR / "unlearning_annotation_results.xlsx"

run_manifest_df = pd.DataFrame([
    {
        "prompt_version": PROMPT_VERSION,
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "base_directory": str(BASE_DIR),
        "codebook_workbook": str(CODEBOOK_WORKBOOK_PATH),
        "codebook_sheet": CODEBOOK_SHEET,
        "examples_workbook": str(EXAMPLES_WORKBOOK_PATH),
        "examples_sheet": EXAMPLES_SHEET,
        "input_documents": len(pdf_paths),
        "paragraph_occurrences": len(paragraphs_df),
        "unique_paragraph_texts": len(unique_paragraphs_df),
        "fewshot_examples_used": len(prompt_examples_df),
        "leakage_similarity_threshold": LEAKAGE_SIMILARITY_THRESHOLD,
        "exclude_same_document_examples": EXCLUDE_SAME_DOCUMENT_EXAMPLES,
        "mock_mode": MOCK_MODE,
    }
])

model_manifest_df = pd.DataFrame(MODEL_CONFIGS)

with pd.ExcelWriter(RESULTS_WORKBOOK_PATH, engine="xlsxwriter") as writer:
    paragraphs_df.to_excel(writer, sheet_name="Extracted Paragraphs", index=False)
    page_quality_df.to_excel(writer, sheet_name="Page Quality", index=False)
    prompt_examples_df.to_excel(writer, sheet_name="Few-shot Examples", index=False)
    example_leakage_audit_df.to_excel(writer, sheet_name="Example Leakage Audit", index=False)
    model_predictions_long_df.to_excel(writer, sheet_name="Model Predictions Long", index=False)
    model_predictions_wide_df.to_excel(writer, sheet_name="Model Predictions Wide", index=False)
    consensus_annotations_df.to_excel(writer, sheet_name="Consensus", index=False)
    reliability_overall_df.to_excel(writer, sheet_name="Reliability Overall", index=False)
    reliability_pairwise_df.to_excel(writer, sheet_name="Reliability Pairwise", index=False)
    reliability_by_document_df.to_excel(writer, sheet_name="Reliability by Doc", index=False)
    run_manifest_df.to_excel(writer, sheet_name="Run Manifest", index=False)
    model_manifest_df.to_excel(writer, sheet_name="Model Manifest", index=False)
    codebook_df.to_excel(writer, sheet_name="Codebook Used", index=False)

    workbook = writer.book
    header_format = workbook.add_format({
        "bold": True, "font_color": "white", "bg_color": "#1F4E78",
        "border": 1, "text_wrap": True, "valign": "top",
    })
    yes_format = workbook.add_format({"bg_color": "#C6EFCE", "font_color": "#006100"})
    disagreement_format = workbook.add_format({"bg_color": "#FFF2CC", "font_color": "#7F6000"})

    for sheet_name, df in [
        ("Extracted Paragraphs", paragraphs_df),
        ("Page Quality", page_quality_df),
        ("Few-shot Examples", prompt_examples_df),
        ("Example Leakage Audit", example_leakage_audit_df),
        ("Model Predictions Long", model_predictions_long_df),
        ("Model Predictions Wide", model_predictions_wide_df),
        ("Consensus", consensus_annotations_df),
        ("Reliability Overall", reliability_overall_df),
        ("Reliability Pairwise", reliability_pairwise_df),
        ("Reliability by Doc", reliability_by_document_df),
        ("Run Manifest", run_manifest_df),
        ("Model Manifest", model_manifest_df),
        ("Codebook Used", codebook_df),
    ]:
        ws = writer.sheets[sheet_name]
        ws.freeze_panes(1, 0)
        ws.autofilter(0, 0, max(len(df), 1), max(len(df.columns) - 1, 0))
        for col_idx, column in enumerate(df.columns):
            ws.write(0, col_idx, column, header_format)
            sample = [len(str(column))] + [len(str(x)) for x in df[column].head(100).fillna("")]
            width = min(max(sample) + 2, 45)
            if column.lower() in {"text", "rationale", "consensus_rationale", "text_preview"} or "rationale" in column.lower():
                width = 45
            ws.set_column(col_idx, col_idx, max(10, width))

        if sheet_name == "Consensus" and not df.empty:
            consensus_col = df.columns.get_loc("consensus_unlearning")
            bucket_col = df.columns.get_loc("agreement_bucket")
            ws.conditional_format(1, consensus_col, len(df), consensus_col, {
                "type": "cell", "criteria": "==", "value": True, "format": yes_format,
            })
            ws.conditional_format(1, bucket_col, len(df), bucket_col, {
                "type": "text", "criteria": "containing", "value": "majority", "format": disagreement_format,
            })

print("Saved:", RESULTS_WORKBOOK_PATH)

Saved: /content/unlearning_pipeline/outputs/unlearning_annotation_results.xlsx


## 15. Add editable highlights and popup comments to each source PDF

Color meaning:

- **Green:** all three models say Yes.
- **Yellow:** two models say Yes.
- **Red/pink:** one model says Yes and two say No; consensus is No but the paragraph is highlighted for review.
- **Blue:** unanimous No, only when `ANNOTATE_UNANIMOUS_NO=True`.

The original PDF is never overwritten.

In [20]:
ANNOTATION_COLORS = {
    "unanimous_yes": (0.35, 0.85, 0.35),
    "majority_yes": (1.00, 0.82, 0.20),
    "majority_no_one_positive": (1.00, 0.45, 0.45),
    "unanimous_no": (0.55, 0.75, 1.00),
}


def yn(value) -> str:
    return "Yes" if bool(value) else "No"


def model_comment_line(row: pd.Series, provider: str) -> list[str]:
    label = yn(row[f"{provider}_unlearning"])
    target = TARGET_DISPLAY.get(row[f"{provider}_target_type"], row[f"{provider}_target_type"])
    confidence = row.get(f"{provider}_confidence", np.nan)
    confidence_text = f"{float(confidence):.0%}" if pd.notna(confidence) else "n/a"
    agency = clean_text(row.get(f"{provider}_agency", "")) or "Not stated"
    rationale = clean_text(row.get(f"{provider}_rationale", ""))
    model = row.get(f"{provider}_model", "")
    strategy = row.get(f"{provider}_strategy", "")
    return [
        f"{provider.upper()} - {model} - {strategy}",
        f"Label: {label} | Target: {target} | Confidence: {confidence_text} | Agency: {agency}",
        f"Rationale: {rationale}",
    ]


def build_annotation_comment(row: pd.Series) -> str:
    consensus_conf = row.get("consensus_confidence", np.nan)
    conf_text = f"{float(consensus_conf):.0%}" if pd.notna(consensus_conf) else "n/a"
    lines = [
        f"Paragraph ID: {row['paragraph_id']}",
        f"Page: {row['page_number']}",
        f"Consensus: {yn(row['consensus_unlearning'])} ({row['yes_votes']}/3 Yes votes)",
        f"Agreement: {row['agreement_bucket']}",
        f"Consensus target: {row['consensus_target_display']}",
        f"Consensus agency/agencies: {clean_text(row.get('consensus_agencies', '')) or 'Not stated'}",
        f"Consensus confidence: {conf_text}",
        f"Review priority: {row['review_priority']}",
        "",
    ]
    for provider in PROVIDER_ORDER:
        lines.extend(model_comment_line(row, provider))
        lines.append("")
    return "\n".join(lines)[:MAX_COMMENT_CHARS]


def should_annotate(row: pd.Series) -> bool:
    if row["agreement_bucket"] == "unanimous_no":
        return ANNOTATE_UNANIMOUS_NO
    if ANNOTATE_ANY_POSITIVE_VOTE:
        return row["yes_votes"] >= 1
    return bool(row["consensus_unlearning"])


def annotate_pdf(source_path: Path, annotations: pd.DataFrame, output_path: Path) -> dict:
    source_path = Path(source_path)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    annotation_count = 0
    failures = []

    with fitz.open(source_path) as doc:
        for _, row in annotations.sort_values(["page_number", "block_number"]).iterrows():
            if not should_annotate(row):
                continue
            page_index = int(row["page_number"]) - 1
            if page_index < 0 or page_index >= len(doc):
                failures.append(f"{row['paragraph_id']}: page index out of range")
                continue

            rect_values = json.loads(row["rects_json"])
            rects = [fitz.Rect(*values) for values in rect_values]
            if not rects:
                failures.append(f"{row['paragraph_id']}: no rectangles")
                continue

            page = doc[page_index]
            try:
                annot = page.add_highlight_annot(rects)
                annot.set_colors(stroke=ANNOTATION_COLORS[row["agreement_bucket"]])
                annot.set_opacity(0.38)
                annot.set_info(
                    title="Unlearning Model Ensemble",
                    subject=(
                        f"Consensus {yn(row['consensus_unlearning'])}; "
                        f"{row['consensus_target_display']}; {row['yes_votes']}/3 Yes"
                    ),
                    content=build_annotation_comment(row),
                )
                annot.update()
                annotation_count += 1
            except Exception as exc:
                failures.append(f"{row['paragraph_id']}: {exc}")

        doc.save(output_path, garbage=4, deflate=True)

    return {
        "source_pdf": str(source_path),
        "annotated_pdf": str(output_path),
        "annotations_added": annotation_count,
        "annotation_failures": len(failures),
        "failure_details": " | ".join(failures[:20]),
    }


annotation_audit_rows = []
for source_file, subset in consensus_annotations_df.groupby("source_file"):
    source_path = Path(source_file)
    output_path = ANNOTATED_DIR / f"{source_path.stem}_unlearning_annotated.pdf"
    annotation_audit_rows.append(annotate_pdf(source_path, subset, output_path))

annotation_audit_df = pd.DataFrame(annotation_audit_rows)
annotation_audit_df.to_csv(OUTPUT_DIR / "annotation_audit.csv", index=False)
display(annotation_audit_df)

if annotation_audit_df["annotation_failures"].sum() > 0:
    warnings.warn("Some PDF annotations failed. Review annotation_audit.csv before using the PDFs.")

,source_pdf,annotated_pdf,annotations_added,annotation_failures,failure_details
0,/content/unlearning_pipeline/input_documents/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of.pdf,/content/unlearning_pipeline/outputs/annotated_pdfs/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of_...,18,0,


## 16. Structural verification and output archive

The verification reopens every annotated PDF and counts annotations. It also confirms that all three providers produced exactly one label per unique paragraph text.

In [21]:
def count_pdf_annotations(pdf_path: Path) -> int:
    total = 0
    with fitz.open(pdf_path) as doc:
        for page in doc:
            annot = page.first_annot
            while annot:
                total += 1
                annot = annot.next
    return total


verification_rows = []
for _, row in annotation_audit_df.iterrows():
    pdf_path = Path(row["annotated_pdf"])
    verification_rows.append({
        "annotated_pdf": str(pdf_path),
        "file_exists": pdf_path.exists(),
        "file_size_bytes": pdf_path.stat().st_size if pdf_path.exists() else 0,
        "annotations_reported": row["annotations_added"],
        "annotations_reopened": count_pdf_annotations(pdf_path) if pdf_path.exists() else 0,
    })

verification_df = pd.DataFrame(verification_rows)
verification_df["annotation_count_matches"] = (
    verification_df["annotations_reported"] == verification_df["annotations_reopened"]
)
verification_df.to_csv(OUTPUT_DIR / "output_verification.csv", index=False)

if not verification_df["file_exists"].all():
    raise RuntimeError("One or more annotated PDF outputs are missing.")
if not verification_df["annotation_count_matches"].all():
    raise RuntimeError("Annotation count verification failed for one or more PDFs.")

expected_prediction_rows = len(unique_paragraphs_df) * len(MODEL_CONFIGS)
actual_prediction_rows = len(valid_unique_predictions_df)
if actual_prediction_rows != expected_prediction_rows:
    raise RuntimeError(
        f"Expected {expected_prediction_rows} successful unique-text predictions; found {actual_prediction_rows}."
    )

archive_base = BASE_DIR / "unlearning_document_annotation_outputs"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_DIR))

display(verification_df)
print("\nCompleted outputs:")
print(" - Results workbook:", RESULTS_WORKBOOK_PATH)
print(" - Annotated PDFs:", ANNOTATED_DIR)
print(" - ZIP archive:", archive_path)

,annotated_pdf,file_exists,file_size_bytes,annotations_reported,annotations_reopened,annotation_count_matches
0,/content/unlearning_pipeline/outputs/annotated_pdfs/Public Administration Review - 2010 - McGuire - What if Hurricane Katrina Hit in 2020 The Need for Strategic Management of_...,True,616628,18,18,True



Completed outputs:
 - Results workbook: /content/unlearning_pipeline/outputs/unlearning_annotation_results.xlsx
 - Annotated PDFs: /content/unlearning_pipeline/outputs/annotated_pdfs
 - ZIP archive: /content/unlearning_pipeline/unlearning_document_annotation_outputs.zip


## Interpretation notes

- Inter-model reliability is **not accuracy**. High agreement can coexist with shared errors.
- Fleiss' kappa may appear low when one class dominates; percent agreement and Krippendorff's alpha are reported alongside it.
- Paragraph extraction should be reviewed before a large production run, especially for multi-column, table-heavy, or scanned documents.
- The examples used by OpenAI and Anthropic are saved in `fewshot_examples_used.csv`, and every excluded overlap is documented in `example_leakage_audit.csv`.
- A one-model Yes / two-model No paragraph is highlighted red for adjudication but retains a consensus No label.